# J1S3 — EDA & Plotly : Exploration Visuelle des Données Credit Risk

**Formation Data Science 3 Jours** | Jour 1 · Session 3

---

## Contexte

Nous repartons du fichier `credit_features_j1.parquet` produit en J1S2 (16 colonnes).
L'objectif est de **transformer les analyses numériques de J1S2 en visualisations** :
- Groupby grade : G=98%, F=71% → histogrammes et box plots
- Quartiles de revenu : rupture Q1→Q2 de 18.6 pts → bar charts
- 68 dossiers F/G+ratio>0.3, 88.2% défaut → scatter pour confirmer visuellement

**Livrable de session** : 8 figures interactives HTML sauvegardées dans `figures/`

> 💡 **Fil rouge** : les insights visuels justifient les features créées en J1S2
> et annoncent le modèle de scoring J2S2 (régression logistique, seuil 0.30).


## Setup — Installation & Imports

In [ ]:
# ── Détection automatique du ROOT du projet ──────────────────────────────────
from pathlib import Path
import os, subprocess, sys

def find_root(markers=('requirements.txt', '.git', 'data')):
    """Remonte l'arborescence jusqu'à trouver la racine du projet."""
    cwd = Path.cwd()
    for folder in [cwd] + list(cwd.parents):
        if any((folder / m).exists() for m in markers):
            return folder
    return cwd

try:
    import google.colab
    IN_COLAB = True
    ROOT = find_root()
except ImportError:
    IN_COLAB = False
    ROOT = find_root()

os.chdir(ROOT)
print(f"ROOT  : {ROOT}")
print(f"Colab : {IN_COLAB}")

# ── Installation depuis requirements.txt ─────────────────────────────────────
req_path = ROOT / 'requirements.txt'
if req_path.exists():
    print(f"\n📦 Installation depuis {req_path} ...")
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', str(req_path), '-q'],
        capture_output=True, text=True
    )
    print("✅ requirements.txt installé" if result.returncode == 0 else f"⚠️  Erreur : {result.stderr[-300:]}")
else:
    print("⚠️  requirements.txt non trouvé — installation du minimum nécessaire")
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'plotly', 'pandas', 'pyarrow', '-q'], check=True)
    print("✅ Packages minimum installés")

# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly

# Créer le dossier figures si nécessaire
os.makedirs(ROOT / 'figures', exist_ok=True)

print(f"✅ Imports OK | Plotly {plotly.__version__}")


In [14]:
# Charger le livrable J1S2
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'credit_features_j1.parquet')

print(f"Shape : {df.shape}")
print(f"Taux de défaut : {df['loan_status'].mean():.1%}")
print("\nColonnes :")
print(df.columns.tolist())


Shape : (32581, 18)
Taux de défaut : 21.8%

Colonnes :
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_to_income', 'credit_risk_score', 'age_bucket', 'income_quartile', 'taux_eleve', 'person_age_clean']


### ✅ Vérification du chargement

**Résultats attendus :**
- Shape : `(32581, 16)` — les 12 colonnes originales + 4 features engineering
- Taux de défaut : `21.8%` — référence pour toute l'EDA
- Colonnes engineering présentes : `loan_to_income`, `credit_risk_score`, `age_bucket`, `income_quartile`

> **Interprétation métier** : 21.8% de défaut = 1 emprunteur sur 5.
> Tout segment avec un taux > 21.8% est en surrisque par rapport à la moyenne du portefeuille.


---
## Bloc 1 — Distributions : Histogrammes & Box Plots

**Durée : 45 min**

### 1.1 Histogramme — Distribution des revenus par statut de prêt

In [15]:
# Distribution des revenus : sains vs défauts
# ── Deux vues complémentaires ────────────────────────────────────────────────
import numpy as np

p99 = df['person_income'].quantile(0.99)
print(f"P99 person_income : {p99:,.0f} $  (max = {df['person_income'].max():,.0f} $)")

# Vue 1 : axe limité à P99 — forme de la distribution principale
fig1a = px.histogram(
    df,
    x='person_income',
    color='loan_status',
    barmode='overlay',
    nbins=80,
    range_x=[0, p99],
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    labels={'loan_status': 'Défaut', 'person_income': 'Revenu annuel ($)'},
    title=f'Distribution des revenus — zoom 0–{p99/1000:.0f}k$ (P99, hors outliers extrêmes)'
)
fig1a.update_layout(bargap=0.02, plot_bgcolor='#021B2E',
                    paper_bgcolor='#021B2E', font_color='white')
fig1a.write_html(str(ROOT / 'figures' / 'hist_income.html'))
fig1a.show()

# Vue 2 : log-transformation de la variable AVANT l'histogramme
# (log_x=True dans px.histogram calcule les bins en linéaire puis applique l'axe log
#  → barres vides. Solution : travailler sur log1p(income) directement)
df_plot = df.copy()
df_plot['log_income'] = np.log1p(df_plot['person_income'])

fig1b = px.histogram(
    df_plot,
    x='log_income',
    color='loan_status',
    barmode='overlay',
    nbins=80,
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    labels={'loan_status': 'Défaut', 'log_income': 'log(Revenu annuel + 1)'},
    title='Distribution des revenus — échelle log1p (toutes valeurs, outliers inclus)'
)
fig1b.update_layout(bargap=0.02, plot_bgcolor='#021B2E',
                    paper_bgcolor='#021B2E', font_color='white')
fig1b.write_html(str(ROOT / 'figures' / 'hist_income_log.html'))
fig1b.show()

print("✅ figures/hist_income.html sauvegardé (zoom P99)")
print("✅ figures/hist_income_log.html sauvegardé (log1p)")


P99 person_income : 225,200 $  (max = 6,000,000 $)


✅ figures/hist_income.html sauvegardé (zoom P99)
✅ figures/hist_income_log.html sauvegardé (log1p)


### 📊 Interprétation — Histogramme revenus

**Pourquoi limiter à P99 ?**
`person_income` max = **6 000 000 $** (outlier J1S2, outlier iloc identifié). Sans limite d'axe,
20 000 barres s'écrasent dans la première colonne — le graphique est illisible.
`range_x=[0, P99]` exclut < 1% des données et révèle la forme réelle.

**Vue 1 — zoom P99 : ce que vous devez voir :**
- Distribution log-normale : concentration entre **20 000 $ et 80 000 $**, queue à droite
- Les défauts (vert) se concentrent davantage sur les **revenus < 40 000 $**
- Forte zone de chevauchement entre 20 000 $ et 60 000 $ → le revenu seul ne suffit pas

**Vue 2 — échelle log : ce que vous devez voir :**
- La distribution s'étale proprement de 4 000 $ à 6 000 000 $
- Les deux courbes (sains/défauts) se superposent mais les défauts dominent les valeurs basses
- La queue droite (revenus > 200 000 $) est presque exclusivement des non-défauts

**Lien J1S2 :** Les 342 dossiers E/F/G+ratio>25% avaient un revenu moyen de **52 587 $** —
situé dans la zone de chevauchement → c'est le RATIO qui discrimine, pas le revenu brut.


### 1.2 Box Plot — Ratio prêt/revenu par grade

In [ ]:
# Box plot : loan_percent_income par grade et statut
fig2 = px.box(
    df,
    x='loan_grade',
    y='loan_percent_income',
    color='loan_status',
    color_discrete_map={0: '#065A82', 1: '#02C39A'},
    labels={
        'loan_grade': 'Grade',
        'loan_percent_income': 'Ratio prêt/revenu',
        'loan_status': 'Défaut'
    },
    title='Ratio prêt/revenu (loan_percent_income) par grade'
)
fig2.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig2.write_html(str(ROOT / 'figures' / 'box_grade.html'))
fig2.show()
print("✅ figures/box_grade.html sauvegardé")


### 📊 Interprétation — Box Plot par grade

**Ce que vous devez voir :**
- Grade A : médiane loan_percent_income ≈ 0.08–0.12, IQR très resserré (emprunteurs sûrs)
- Grade G : médiane ≈ **0.53**, IQR = **[0.42–0.59]** (les défauts dominent)
- Les boîtes des défauts (vert) sont systématiquement plus hautes que les sains (bleu)

**Confirmation J1S2 :** En Bloc 3 J1S2, les 7 108 défauts avaient `loan_percent_income` dominant
entre **0.53 et 0.59**. La médiane grade G = 0.53 **confirme exactement ce filtre**.

> 💡 La visualisation valide la règle numérique. Si votre box plot grade G ne montrait pas
> de médiane autour de 0.53, ce serait le signal d'un problème dans le parquet J1S2.


---
## Bloc 2 — Corrélations : Heatmap & Scatter Matrix

**Durée : 50 min**

### 2.1 Heatmap de corrélations

In [ ]:
# Heatmap de corrélations entre features numériques
num_cols = [
    'person_age', 'person_income', 'loan_amnt',
    'loan_percent_income', 'loan_to_income',
    'credit_risk_score', 'loan_status'
]
corr = df[num_cols].corr().round(2)

fig3 = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto=True,
    title='Matrice de corrélation — features numériques'
)
fig3.update_layout(
    width=700, height=600,
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig3.write_html(str(ROOT / 'figures' / 'heatmap_corr.html'))
fig3.show()

# Afficher les 3 corrélations les plus importantes avec loan_status
print("\nTop corrélations avec loan_status :")
print(corr['loan_status'].sort_values(key=abs, ascending=False)[1:])


### 📊 Interprétation — Heatmap de corrélations

**Corrélations clés avec `loan_status` :**
| Feature | Corrélation | Interprétation |
|---------|------------|----------------|
| `loan_percent_income` | **≈ +0.37** | Plus le ratio est élevé → plus le défaut est probable |
| `loan_to_income` | **≈ +0.33** | Même direction, feature complémentaire |
| `person_income` | **≈ −0.19** | Revenu élevé protège (modérément) |
| `loan_amnt` | **≈ +0.16** | Montants élevés = légèrement plus risqués |
| `person_age` | **≈ −0.03** | Quasi aucun effet linéaire direct |

**Point clé :** `loan_percent_income` (r≈0.37) est la feature la plus discriminante.
C'est ce que les filtres J1S2 avaient révélé numériquement — la heatmap le confirme visuellement.

> ⚠️ `credit_risk_score` corrèle fortement avec `loan_to_income` (multicolinéarité potentielle).
> En modélisation J2S2, on évaluera si garder les deux est utile (VIF).


### 2.2 Scatter Matrix

In [ ]:
# Scatter matrix — 4 features clés
# sample(3000) pour la performance (32k points = navigateur lent)
fig4 = px.scatter_matrix(
    df.sample(3000, random_state=42),
    dimensions=['loan_percent_income', 'loan_to_income',
                'person_income', 'credit_risk_score'],
    color='loan_status',
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    labels={'loan_status': 'Défaut'},
    opacity=0.4,
    title='Scatter matrix — 4 features vs défaut de paiement'
)
fig4.update_traces(marker=dict(size=3))
fig4.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig4.write_html(str(ROOT / 'figures' / 'scatter_matrix.html'))
fig4.show()
print("✅ figures/scatter_matrix.html sauvegardé")


### 📊 Interprétation — Scatter Matrix

**Sur l'axe `loan_percent_income` :**
- Points verts (défauts) décalés vers la droite (valeurs élevées)
- Points bleus (sains) concentrés à gauche
- C'est la **séparation que le modèle logistique J2S2 va apprendre**

**Observation importante :** `loan_percent_income` et `loan_to_income` semblent très corrélés
mais ne sont pas identiques :
- `loan_to_income` = montant total / revenu annuel (taille de la dette)
- `loan_percent_income` = mensualité / revenu (charge courante)

Les deux capturent des aspects différents du risque. Le modèle J2S2 décidera lequel garder.


---
## Bloc 3 — Analyse Bivariée : Risque par Segment

**Durée : 55 min**

### 3.1 Taux de défaut par grade

In [ ]:
# Calcul du taux de défaut par grade
default_by_grade = (
    df.groupby('loan_grade')['loan_status']
      .mean()
      .reset_index()
)
default_by_grade.columns = ['grade', 'taux_defaut']
print(default_by_grade.to_string(index=False))

# Visualisation
fig5 = px.bar(
    default_by_grade,
    x='grade',
    y='taux_defaut',
    color='taux_defaut',
    color_continuous_scale=['#065A82', '#1C7293', '#02C39A'],
    text_auto='.1%',
    labels={'taux_defaut': 'Taux de défaut', 'grade': 'Grade'},
    title='Taux de défaut par grade (A → G)'
)
fig5.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white',
    showlegend=False
)
fig5.write_html(str(ROOT / 'figures' / 'defaut_grade.html'))
fig5.show()
print("✅ figures/defaut_grade.html sauvegardé")


### 📊 Interprétation — Gradient A → G

**Résultats confirmés de J1S2 :**
| Grade | Taux de défaut |
|-------|---------------|
| A | ~7% |
| B | ~12% |
| C | ~28% |
| D | ~45% |
| E | ~61% |
| F | **~71%** |
| G | **~98%** |

**Ce graphique est votre outil de communication métier :**
Un directeur des risques peut prendre une décision en 5 secondes en regardant ce bar chart.

> 💡 **Question de réflexion :** Une règle "grade ≥ E = refus automatique" couvrirait
> les cas les plus évidents. Mais un modèle peut faire mieux : un grade E avec revenu élevé
> et ratio faible est moins risqué qu'un grade D avec ratio élevé. C'est l'objet de J2S2.


### 3.2 Scatter bivarié — loan_to_income vs loan_percent_income par grade

In [ ]:
# Scatter : deux features ratio par grade et statut de défaut
fig6 = px.scatter(
    df.sample(5000, random_state=42),
    x='loan_to_income',
    y='loan_percent_income',
    color='loan_grade',
    symbol='loan_status',
    opacity=0.5,
    labels={
        'loan_to_income': 'Ratio prêt/revenu annuel',
        'loan_percent_income': 'Ratio mensualité/revenu',
        'loan_grade': 'Grade',
        'loan_status': 'Défaut'
    },
    title='loan_to_income vs loan_percent_income — par grade et statut'
)
fig6.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig6.write_html(str(ROOT / 'figures' / 'scatter_features.html'))
fig6.show()
print("✅ figures/scatter_features.html sauvegardé")


### 📊 Interprétation — Scatter bivarié

**Séparation visuelle par grade :**
- Grades A/B (bas à gauche) : faibles ratios, peu de défauts (triangles)
- Grades F/G (haut à droite) : ratios élevés, majorité de défauts

**La frontière n'est pas une droite :**
Les emprunteurs sains et défauts se mélangent dans la zone centrale.
Le modèle logistique J2S2 va apprendre la **frontière de décision optimale**
dans cet espace à deux dimensions (et plus si on ajoute d'autres features).

> 💡 C'est exactement pour ça qu'on a créé les deux features en J1S2 :
> `loan_to_income` ET `loan_percent_income` → chacune apporte un angle différent.


---
## Bloc 4 — Fil Rouge : Features Engineering → Modèle

**Durée : 40 min**

### 4.1 income_quartile — Défaut par quartile de revenu

In [16]:
# Taux de défaut par quartile de revenu
# income_quartile créé avec pd.qcut en J1S2 Bloc 4
defaut_quartile = (
    df.groupby('income_quartile')['loan_status']
      .mean()
      .reset_index()
)
defaut_quartile.columns = ['quartile', 'taux_defaut']
print(defaut_quartile.to_string(index=False))

fig7 = px.bar(
    defaut_quartile,
    x='quartile',
    y='taux_defaut',
    text_auto='.1%',
    color_discrete_sequence=['#1C7293'],
    labels={
        'quartile': 'Quartile de revenu',
        'taux_defaut': 'Taux de défaut'
    },
    title='Taux de défaut par quartile de revenu (income_quartile)'
)
fig7.add_hline(
    y=df['loan_status'].mean(),
    line_dash='dash',
    line_color='#02C39A',
    annotation_text=f"Moyenne : {df['loan_status'].mean():.1%}"
)
fig7.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig7.write_html(str(ROOT / 'figures' / 'defaut_quartile.html'))
fig7.show()
print("✅ figures/defaut_quartile.html sauvegardé")


     quartile  taux_defaut
       Q1_bas     0.396958
 Q2_moyen_bas     0.213218
Q3_moyen_haut     0.170701
      Q4_haut     0.091121


C:\Users\alex.diby\AppData\Local\Temp\ipykernel_13348\3920604483.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('income_quartile')['loan_status']


✅ figures/defaut_quartile.html sauvegardé


### 📊 Interprétation — income_quartile

**La rupture Q1 → Q2 en image :**
- Q1 (revenus les plus bas) : **39.7%** de défaut — quasi 2× la moyenne du portefeuille
- Q2 : **21.3%** de défaut — retour proche de la moyenne
- Q3 et Q4 : taux décroissants, Q4 < 10%

**Ce graphique justifie rétrospectivement** l'utilisation de `pd.qcut` en J1S2 :
la rupture Q1→Q2 de **−18.6 points** est spectaculaire et immédiatement visible.

La ligne en pointillé (moyenne 21.8%) permet de voir rapidement quels quartiles
sont en surrisque (au-dessus) ou sous-risque (en dessous).

> 💡 `income_quartile` sera une feature catégorielle dans le modèle J2S2.
> Son pouvoir discriminant est maintenant visuellement démontré.


### 4.2 age_bucket — Défaut par tranche d'âge

In [17]:
# Distribution des défauts par tranche d'âge
# age_bucket créé avec pd.cut en J1S2 Bloc 4
fig8 = px.histogram(
    df,
    x='age_bucket',
    color='loan_status',
    barmode='group',
    barnorm='percent',
    color_discrete_map={0: '#065A82', 1: '#02C39A'},
    labels={
        'age_bucket': "Tranche d'âge",
        'loan_status': 'Défaut'
    },
    title="Taux de défaut par tranche d'âge (age_bucket)"
)
fig8.update_layout(
    plot_bgcolor='#021B2E',
    paper_bgcolor='#021B2E',
    font_color='white'
)
fig8.write_html(str(ROOT / 'figures' / 'defaut_age.html'))
fig8.show()
print("✅ figures/defaut_age.html sauvegardé")


✅ figures/defaut_age.html sauvegardé


### 📊 Interprétation — age_bucket

**L'effet âge est réel mais modéré :**
- Les 20-30 ans ont un taux légèrement plus élevé (moins d'historique de crédit, emploi moins stable)
- L'effet diminue progressivement avec l'âge
- `barnorm='percent'` permet la comparaison malgré des effectifs différents par tranche

**Par rapport à loan_percent_income (r≈0.37) :**
L'effet de l'âge est moins fort. `age_bucket` enrichit le modèle
mais n'est pas la variable principale.

> 💡 Ces 8 figures constituent votre **dossier EDA complet**.
> Chaque feature créée en J1S2 a maintenant une justification visuelle.
> C'est le cahier des charges du modèle de scoring J2S2.


---
## Vérification des livrables & Commit GitHub

In [18]:
# Vérification : toutes les figures sont-elles présentes ?
figures_attendues = [
    'hist_income', 'hist_income_log', 'box_grade', 'heatmap_corr',
    'scatter_matrix', 'defaut_grade', 'scatter_features',
    'defaut_quartile', 'defaut_age'
]

print("Vérification des livrables J1S3 :")
print("=" * 45)
all_ok = True
for fig in figures_attendues:
    path = ROOT / 'figures' / f'{fig}.html'
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"✅  {path}  ({size_kb:.0f} KB)")
    else:
        print(f"❌  {path}  — MANQUANT")
        all_ok = False

print("=" * 45)
if all_ok:
    print("\n🎉  Tous les livrables sont présents. Prêt pour le commit !")
else:
    print("\n⚠️  Des fichiers manquent — réexécuter les cellules correspondantes.")


Vérification des livrables J1S3 :
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\hist_income.html  (4915 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\hist_income_log.html  (5103 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\box_grade.html  (5284 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\heatmap_corr.html  (4743 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\scatter_matrix.html  (4871 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\defaut_grade.html  (4743 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\scatter_features.html  (4876 KB)
✅  d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\figures\defaut_quartil

In [19]:
# Commit GitHub
# Décommenter pour exécuter

# !git add notebooks/j1s3_eda_plotly.ipynb figures/
# !git commit -m 'feat(j1s3): EDA complet Plotly - 8 figures credit risk'
# !git push origin main

print("Commit à effectuer manuellement ou décommenter les lignes ci-dessus.")
print("Message de commit : feat(j1s3): EDA complet Plotly - 8 figures credit risk")


Commit à effectuer manuellement ou décommenter les lignes ci-dessus.
Message de commit : feat(j1s3): EDA complet Plotly - 8 figures credit risk


---
## ✅ Bilan de Session — J1S3 EDA & Plotly

| Bloc | Réalisé | Figures produites |
|------|---------|------------------|
| 1 — Distributions | ✅ | `hist_income.html`, `hist_income_log.html`, `box_grade.html` |
| 2 — Corrélations | ✅ | `heatmap_corr.html`, `scatter_matrix.html` |
| 3 — Bivarié | ✅ | `defaut_grade.html`, `scatter_features.html` |
| 4 — Fil rouge | ✅ | `defaut_quartile.html`, `defaut_age.html` |

### Ce que l'EDA a confirmé

1. **loan_percent_income** est la feature la plus discriminante (r≈0.37, médiane grade G = 0.53–0.59)
2. **Le gradient A(7%)→G(98%)** est spectaculaire et communicable immédiatement
3. **La rupture Q1→Q2** de -18.6 points justifie `income_quartile` comme feature
4. **Deux features ratio** (loan_to_income ET loan_percent_income) sont complémentaires

### Prochaine étape — J1S4 : Nettoyage & Feature Engineering

Les insights visuels sont validés. Avant de passer au Machine Learning,
le dataset doit être **nettoyé et préparé** : suppression des outliers biologiquement
impossibles (`person_age` > 90, `person_emp_length` > 123), imputation des NaN
(`person_emp_length` 2.7%, `loan_int_rate` 9.5%), encodage des variables catégorielles,
et nouvelles features métier (`debt_to_income`, `log_income`, `emp_stability`).

**Livrable J1S4** : `data/processed/credit_risk_clean.parquet` — dataset ML-ready,
0 NaN, ~32 200 lignes, ~28 features. Point de départ de tout le Jour 2.
